# LLM Data Extraction from PubMed Abstracts

This workbook teaches you how to build a **config-driven data extraction pipeline** using a large language model (LLM). You'll extract structured data from stroke research abstracts — the same pattern used in real clinical research, scaled down so you can understand every piece.

**What you'll build:**
1. A configuration that defines what to extract (no coding required to change)
2. A prompt builder that turns that configuration into instructions for the LLM
3. Test cases with known correct answers
4. A scoring system to measure how well the LLM performed

**Who this is for:** Researchers comfortable with computers but new to coding. We explain Python idioms as they appear.

**How to use this notebook:** Run each cell in order by clicking on it and pressing `Shift+Enter`. If something goes wrong, go to **Kernel > Restart & Run All** to start fresh.

**Cost:** This notebook makes 4 API calls to Claude. Total cost is roughly $0.01–0.05. Each time you re-run a cell that calls the API, you'll make (and pay for) another call.

## 1. Prerequisites

Before running this notebook, you need three things:

1. **Python 3.9+** — [Installation guide](https://www.python.org/downloads/)
2. **The `anthropic` package** — Run `pip install anthropic` in your terminal before launching Jupyter
3. **An Anthropic API key** — [Create one here](https://console.anthropic.com/settings/keys)

### Setting your API key securely

Your API key is like a password — **never paste it directly into this notebook.** If you share the notebook or commit it to git, the key goes with it.

Instead, set it as an environment variable in your terminal *before* launching Jupyter:

```bash
export ANTHROPIC_API_KEY="sk-ant-..."
jupyter notebook
```

The cell below checks that everything is in place.

In [1]:
# Prerequisites check — run this first
import sys

# Check Python version
assert sys.version_info >= (3, 9), f"Python 3.9+ required, you have {sys.version}"
print(f"Python {sys.version.split()[0]} — OK")

# Check that the anthropic package is installed
try:
    import anthropic
    print(f"anthropic package — OK")
except ImportError:
    raise ImportError("The anthropic package is not installed. Run: pip install anthropic")

# Check that the API key is set
import os
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not set. "
    "Set it in your terminal before launching Jupyter: "
    "export ANTHROPIC_API_KEY='sk-ant-...'"
)
print("API key — OK")

print("\nAll prerequisites met. You're ready to go.")

Python 3.14.3 — OK
anthropic package — OK
API key — OK

All prerequisites met. You're ready to go.


## 2. Setup

We need two things to talk to the LLM:

- **`anthropic`** — the Python package that communicates with Claude (the LLM we'll use)
- **`json`** — a built-in Python module for working with structured data. JSON (JavaScript Object Notation) is how we'll ask the LLM to format its answers — it's a standard way to represent structured data that both humans and computers can read.

The `anthropic.Anthropic()` call creates a **client** — think of it as opening a phone line to Claude. It automatically uses the API key you set in your environment.

In [2]:
import anthropic
import json
import time

# Create the client — this is your connection to Claude
client = anthropic.Anthropic()

# The model we'll use for extraction
MODEL = "claude-sonnet-4-6"

print(f"Client ready, using model: {MODEL}")

Client ready, using model: claude-sonnet-4-6


## 3. Configuration

This is the most important design idea in the whole workbook: **the extraction schema is data, not code.**

The `CONFIG` below is a Python dictionary (or "dict") — a collection of key-value pairs, written with curly braces `{}`. It defines:
- What fields to extract from each abstract
- What values are valid for each field
- Which fields are required vs. optional

**Why this matters:** When you move to your own research domain, you change this configuration — not the code. A collaborator who doesn't write Python can still define what to extract by editing this dict.

### A note on field selection

We deliberately chose fields that have **unambiguous correct answers** so we can score the LLM's accuracy automatically with exact matching. In a real pipeline, you'd include fuzzier fields too (e.g., "key limitation of the study") — but those need a different scoring approach.

Notice that some of these fields could be extracted with simple text search (grep/regex):
- **`study_type`** is usually stated explicitly in the abstract ("randomized trial", "cohort study")
- **`sample_size`** often appears as a number near words like "patients" or "enrolled"

Other fields genuinely need an LLM — they require reading comprehension and inference:
- **`intervention`** may be described across multiple sentences with varying terminology
- **`primary_outcome_direction`** requires synthesizing the results and conclusion
- **`population_age_group`** sometimes must be inferred from context like "mean age 72"
- **`single_center`** is sometimes implied ("patients at our hospital") rather than stated

In [3]:
CONFIG = {
    "fields": {
        "study_type": {
            "type": "enum",
            "values": ["RCT", "cohort", "case-control", "meta-analysis", "case_report", "other"],
            "description": "The study design. Use 'RCT' for randomized controlled trials.",
            "required": True
        },
        "sample_size": {
            "type": "integer",
            "description": "Total number of participants enrolled. For meta-analyses, use the total pooled N across all included studies.",
            "required": False,
            "nullable": True
        },
        "intervention": {
            "type": "enum",
            "values": ["thrombectomy", "aspirin", "cathodal_tDCS", "anticoagulation", "tPA", "statin", "other"],
            "description": "The primary treatment or exposure studied. Choose the closest match from the list.",
            "required": True
        },
        "primary_outcome_direction": {
            "type": "enum",
            "values": ["positive", "negative", "neutral"],
            "description": "Did the intervention show statistically significant benefit (positive), no significant benefit or harm (negative), or mixed/inconclusive results across subgroups (neutral)?",
            "required": True
        },
        "population_age_group": {
            "type": "enum",
            "values": ["pediatric", "adult", "elderly", "mixed", "not_specified"],
            "description": "The age group of the study population. Use 'elderly' if mean/median age >=65 or described as elderly/older. Use 'adult' if clearly adults but not elderly. Use 'not_specified' only if the abstract provides no age information at all.",
            "required": True
        },
        "single_center": {
            "type": "boolean",
            "description": "Was the study conducted at a single center/institution? False if multi-center, a collaborative group, or a meta-analysis.",
            "required": True
        }
    }
}

# Show what we defined
print(f"Extraction schema defines {len(CONFIG['fields'])} fields:")
for name, spec in CONFIG["fields"].items():
    required = "required" if spec.get("required") else "optional"
    print(f"  {name} ({spec['type']}, {required})")

Extraction schema defines 6 fields:
  study_type (enum, required)
  sample_size (integer, optional)
  intervention (enum, required)
  primary_outcome_direction (enum, required)
  population_age_group (enum, required)
  single_center (boolean, required)


## 4. Prompt Builder

Now we need to turn the configuration into instructions the LLM can follow. This function builds a **prompt** — the text we send to Claude — dynamically from whatever fields are in `CONFIG`.

This is where the config-driven design pays off: if you add a new field to `CONFIG`, it automatically appears in the prompt. You don't need to rewrite any instructions.

The function uses **f-strings** — Python's way of inserting variables into text. The syntax `f"...{variable}..."` replaces `{variable}` with its value. For example, `f"Hello {name}"` becomes `"Hello Alice"` if `name` is `"Alice"`.

In [4]:
def build_prompt(config, abstract_text):
    """
    Build an extraction prompt from the config and an abstract.
    
    This function reads the CONFIG to know what fields to ask for,
    then writes instructions that tell the LLM exactly how to respond.
    """
    # Build a description of each field for the LLM
    field_descriptions = []
    for name, spec in config["fields"].items():
        desc = f"- **{name}** ({spec['type']}): {spec['description']}"
        if spec["type"] == "enum":
            desc += f" Valid values: {spec['values']}"
        if spec.get("nullable"):
            desc += " Use null if not stated."
        field_descriptions.append(desc)
    
    # Combine into the full prompt
    # The triple-quotes (\"\"\"...\"\"\") let us write multi-line strings in Python
    prompt = f"""Extract structured data from the following PubMed abstract.

## Fields to extract

{chr(10).join(field_descriptions)}

## Instructions

- Return ONLY a JSON object with the field names as keys
- Use exactly the field names and value formats specified above
- For enum fields, use exactly one of the listed valid values
- Do not include any text before or after the JSON object

## Abstract

{abstract_text}"""
    
    return prompt


# Let's see what the prompt looks like for a short example
example_prompt = build_prompt(CONFIG, "[abstract text would go here]")
print(example_prompt)

Extract structured data from the following PubMed abstract.

## Fields to extract

- **study_type** (enum): The study design. Use 'RCT' for randomized controlled trials. Valid values: ['RCT', 'cohort', 'case-control', 'meta-analysis', 'case_report', 'other']
- **sample_size** (integer): Total number of participants enrolled. For meta-analyses, use the total pooled N across all included studies. Use null if not stated.
- **intervention** (enum): The primary treatment or exposure studied. Choose the closest match from the list. Valid values: ['thrombectomy', 'aspirin', 'cathodal_tDCS', 'anticoagulation', 'tPA', 'statin', 'other']
- **primary_outcome_direction** (enum): Did the intervention show statistically significant benefit (positive), no significant benefit or harm (negative), or mixed/inconclusive results across subgroups (neutral)? Valid values: ['positive', 'negative', 'neutral']
- **population_age_group** (enum): The age group of the study population. Use 'elderly' if mean/media

**What you just learned:** The prompt is generated from the config, not written by hand. The `build_prompt` function loops over every field in `CONFIG["fields"]` and writes instructions for each one. If you add a field to the config, it will appear in the prompt automatically.

## 5. Test Abstracts

To measure whether the LLM extracts data correctly, we need **test cases** — abstracts where we already know the right answers. This is the same idea as an answer key for an exam.

We've selected 4 real PubMed abstracts from stroke research, each chosen to test a different extraction challenge:

| # | Study | Challenge |
|---|-------|----------|
| 1 | MR CLEAN (thrombectomy RCT) | Baseline — all fields clearly stated |
| 2 | CHS aspirin cohort | Distinguish cohort from RCT; infer "elderly" from age stats |
| 3 | STICA (brain stimulation pilot) | Small single-center trial with negative result |
| 4 | ESUS anticoagulation meta-analysis | Neutral overall result despite subgroup effects |

Each test case is a Python **dictionary** with three keys:
- `"pmid"` — the PubMed ID so you can look it up
- `"abstract"` — the full abstract text
- `"expected"` — the correct extraction (our answer key)

In [5]:
TEST_CASES = [
    {
        "name": "MR CLEAN — thrombectomy RCT",
        "pmid": "25517348",
        "abstract": (
            "Background: In patients with acute ischemic stroke caused by a proximal "
            "intracranial arterial occlusion, intraarterial treatment is highly effective "
            "for emergency revascularization. However, proof of a beneficial effect on "
            "functional outcome is lacking. "
            "Methods: We conducted a randomized clinical trial at 16 medical centers in "
            "the Netherlands. We assigned 500 patients who had acute ischemic stroke "
            "caused by a proximal intracranial occlusion of the anterior circulation to "
            "receive intraarterial treatment plus usual care or usual care alone. "
            "Intraarterial treatment consisted primarily of intraarterial thrombectomy. "
            "The primary outcome was the score on the modified Rankin scale at 90 days. "
            "Results: The adjusted common odds ratio was 1.67 (95% confidence interval "
            "[CI], 1.21 to 2.30). An absolute difference of 13.5 percentage points "
            "favored intervention for functional independence rates (32.6% versus 19.1%). "
            "There were no significant differences in mortality or the occurrence of "
            "symptomatic intracerebral hemorrhage. "
            "Conclusions: In patients with acute ischemic stroke caused by a proximal "
            "intracranial occlusion of the anterior circulation, intraarterial treatment "
            "administered within 6 hours after stroke onset was effective and safe."
        ),
        "expected": {
            "study_type": "RCT",
            "sample_size": 500,
            "intervention": "thrombectomy",
            "primary_outcome_direction": "positive",
            "population_age_group": "not_specified",
            "single_center": False
        }
    },
    {
        "name": "CHS — aspirin and stroke in the elderly",
        "pmid": "9596230",
        "abstract": (
            "Background and purpose: Randomized clinical trials testing aspirin in "
            "relatively low-risk, middle-aged people have consistently shown small "
            "increases in stroke associated with aspirin use. We analyzed the relationship "
            "between the regular use of aspirin and incident ischemic and hemorrhagic "
            "stroke among people aged 65 years or older participating in the "
            "Cardiovascular Health Study. "
            "Methods: We conducted a multivariate analysis of incident stroke rates in a "
            "prospectively assessed, observational cohort of 5011 elderly people followed "
            "for a mean of 4.2 years. "
            "Results: Participants had a mean age of 72 years, and 58% were women. "
            "Twenty-three percent used aspirin frequently, and 17% used aspirin "
            "infrequently at study entry. Frequent aspirin use was associated with an "
            "increased rate of ischemic stroke compared with nonusers (relative risk=1.6; "
            "95% confidence interval [CI], 1.2 to 2.2; P=0.001). After adjustment for "
            "other stroke risk factors, women who used aspirin frequently or infrequently "
            "at study entry had a 1.8-fold (95% CI, 1.2 to 2.8) and 1.6-fold (95% CI, "
            "0.9 to 3.0) increased risk of ischemic stroke, respectively (P<0.01, test "
            "for trend), compared with nonusers. In men, aspirin use was not statistically "
            "significantly associated with stroke risk. Aspirin use at entry was also "
            "associated with a 4-fold (95% CI, 1.6 to 10.0) increase in risk of "
            "hemorrhagic stroke for both infrequent and frequent users of aspirin "
            "(P=0.003). "
            "Conclusions: Aspirin use was associated with increased risks of ischemic "
            "stroke in women and hemorrhagic stroke overall in this elderly cohort, after "
            "adjustment for other stroke predictors. The possibility exists of confounding "
            "by reasons for aspirin use rather than cause and effect. Whether regular "
            "aspirin use increases stroke risk for elderly people without cardiovascular "
            "disease can only be determined by randomized clinical trials."
        ),
        "expected": {
            "study_type": "cohort",
            "sample_size": 5011,
            "intervention": "aspirin",
            "primary_outcome_direction": "negative",
            "population_age_group": "elderly",
            "single_center": False
        }
    },
    {
        "name": "STICA — cathodal tDCS pilot trial",
        "pmid": "33866820",
        "abstract": (
            "Background and purpose: In acute stroke, preventing infarct growth until "
            "complete recanalization occurs is a promising approach as an adjunct to "
            "reperfusion therapies to reduce infarct size and improve outcome. In rodent "
            "models, cathodal transcranial direct current stimulation (C-tDCS) decreases "
            "peri-infarct depolarizations and reduces infarct volume. We hypothesized that "
            "C-tDCS would nonpharmacologically reduce infarct growth in hyperacute middle "
            "cerebral artery territory stroke patients receiving reperfusion therapy. "
            "Methods: STICA was a pilot single-center, double-blind, 2-arms 1:1 "
            "randomized trial evaluating the safety, feasibility, and efficacy of C-tDCS "
            "versus sham stimulation in patients eligible for recanalization therapies. "
            "Magnetic resonance imaging was obtained both on admission and 24 hours later. "
            "The primary end point was 24-hour infarct growth. Secondary outcomes were "
            "National Institutes of Health Stroke Scale score difference between day 7 and "
            "admission and 3-month modified Rankin Scale score. "
            "Results: Forty-five patients were randomized. Median magnetic "
            "resonance imaging-to-C-tDCS start time was 45 minutes; C-tDCS was started "
            "before completion of recanalization procedure in all patients. The "
            "intervention proved feasible in all patients. No major adverse effects "
            "occurred in either group. There was no significant difference between active "
            "and sham groups for any end point. "
            "Conclusions: C-tDCS was feasible and well tolerated. No significant "
            "difference was found between the active and sham groups."
        ),
        "expected": {
            "study_type": "RCT",
            "sample_size": 45,
            "intervention": "cathodal_tDCS",
            "primary_outcome_direction": "negative",
            "population_age_group": "adult",
            "single_center": True
        }
    },
    {
        "name": "ESUS — anticoagulation vs antiplatelets meta-analysis",
        "pmid": "39365971",
        "abstract": (
            "Background and objectives: The term 'embolic stroke of undetermined source' "
            "(ESUS) encompasses a substantial but heterogeneous population of patients "
            "with ischemic stroke, underscoring the importance of identifying personalized "
            "treatment strategies. In subgroups of patients randomized in ESUS trials, we "
            "evaluated the effectiveness of anticoagulation compared with antiplatelet "
            "therapy in secondary ischemic stroke prevention. "
            "Methods: A study-level meta-analysis was conducted on randomized controlled "
            "trials of patients with ESUS, comparing anticoagulation with antiplatelet "
            "therapy. The primary efficacy outcome was recurrent ischemic stroke, and "
            "safety outcomes were major bleeding and death. Subgroups assessed were age, "
            "sex, presence of patent foramen ovale (PFO), left atrial enlargement (LAE), "
            "and atrial cardiopathy. Pooled risk ratios (RRs) were meta-analyzed. "
            "Results: A total of 7 randomized controlled trials involving 14,804 patients "
            "were analyzed, with 7,406 patients treated with anticoagulation and 7,398 "
            "treated with antiplatelet therapy. Compared with antiplatelet therapy, "
            "anticoagulation was associated with a similar rate of recurrent ischemic "
            "stroke (RR 0.91, 95% CI 0.80-1.05). In ESUS with PFO, anticoagulation was "
            "associated with significantly lower risk of ischemic stroke (RR 0.59, 95% CI "
            "0.35-0.98). Subgroups based on age, sex, or presence of atrial cardiopathy "
            "did not benefit from anticoagulation over antiplatelet therapy. "
            "Discussion: In this meta-analysis, an empiric anticoagulation approach is not "
            "beneficial for patients with ESUS. This finding highlights the importance of "
            "an individualized treatment strategy. Anticoagulation treatment showed "
            "promise in patients with medically treated PFO. Other subgroups did not "
            "benefit from anticoagulation therapy."
        ),
        "expected": {
            "study_type": "meta-analysis",
            "sample_size": 14804,
            "intervention": "anticoagulation",
            "primary_outcome_direction": "neutral",
            "population_age_group": "adult",
            "single_center": False
        }
    }
]

print(f"Loaded {len(TEST_CASES)} test cases:")
for i, tc in enumerate(TEST_CASES, 1):
    print(f"  {i}. {tc['name']} (PMID {tc['pmid']})")

Loaded 4 test cases:
  1. MR CLEAN — thrombectomy RCT (PMID 25517348)
  2. CHS — aspirin and stroke in the elderly (PMID 9596230)
  3. STICA — cathodal tDCS pilot trial (PMID 33866820)
  4. ESUS — anticoagulation vs antiplatelets meta-analysis (PMID 39365971)


**What you just learned:** Test cases are the foundation of any extraction pipeline. Without known correct answers, you can't measure whether the LLM is actually doing a good job. In a real study, you'd create these by having domain experts manually extract data from a sample of documents.

## 6. Scoring

Now we need a way to compare the LLM's output against our answer key. We use **exact matching** — either the LLM got the field right or it didn't.

This is a deliberate simplification. We chose fields with unambiguous correct answers specifically so exact matching works. In a production pipeline with fuzzier fields (e.g., free-text summaries), you'd use more sophisticated scoring — semantic similarity, human review, or LLM-as-judge approaches.

In [6]:
def score_extraction(expected, actual):
    """
    Compare expected and actual extractions field by field.
    
    Returns a dict mapping each field name to True (correct) or False (incorrect).
    """
    results = {}
    for field_name in expected:
        expected_val = expected[field_name]
        actual_val = actual.get(field_name)  # .get() returns None if key is missing
        
        # For string comparisons, normalize to lowercase to avoid case mismatches
        if isinstance(expected_val, str) and isinstance(actual_val, str):
            results[field_name] = expected_val.lower() == actual_val.lower()
        else:
            results[field_name] = expected_val == actual_val
    
    return results


def summarize_scores(all_scores):
    """
    Compute overall accuracy and per-field accuracy from a list of score dicts.
    """
    # Count correct and total across all test cases and fields
    total_correct = 0
    total_fields = 0
    
    # Track per-field accuracy
    # A dict of lists: {"study_type": [True, True, False, True], ...}
    field_results = {}
    
    for scores in all_scores:
        for field_name, correct in scores.items():
            total_fields += 1
            if correct:
                total_correct += 1
            if field_name not in field_results:
                field_results[field_name] = []
            field_results[field_name].append(correct)
    
    overall = total_correct / total_fields if total_fields > 0 else 0
    
    per_field = {}
    for field_name, results in field_results.items():
        per_field[field_name] = sum(results) / len(results)
    
    return overall, per_field


# Quick test: score a perfect extraction against itself
test_score = score_extraction(
    TEST_CASES[0]["expected"],
    TEST_CASES[0]["expected"]
)
print(f"Self-score test (should be all True): {test_score}")

Self-score test (should be all True): {'study_type': True, 'sample_size': True, 'intervention': True, 'primary_outcome_direction': True, 'population_age_group': True, 'single_center': True}


**What you just learned:** The scoring functions are separate from the extraction logic. This separation lets you swap in different scoring approaches later (fuzzy matching, partial credit) without changing how extraction works.

## 7. Running the Pipeline

Now we put it all together. For each test abstract, we:

1. **Build the prompt** from config + abstract text
2. **Call the API** to send the prompt to Claude
3. **Parse the response** — Claude returns text, and we need to extract the JSON from it
4. **Score the extraction** against our answer key

The `run_experiment` function below orchestrates this pipeline. It also tracks how long each API call takes, which matters when you're budgeting time for a large extraction job.

**Note on JSON parsing:** LLMs sometimes wrap their JSON in markdown code fences (`` ```json ... ``` ``). The code below strips those if present, which makes the parsing more robust.

In [7]:
def call_llm(client, model, prompt):
    """
    Send a prompt to Claude and return the response text.
    """
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    # The response contains a list of content blocks; we want the text from the first one
    return response.content[0].text


def parse_json_response(text):
    """
    Parse JSON from the LLM's response, handling markdown code fences.
    
    LLMs sometimes wrap JSON in ```json ... ``` markers.
    This function strips those before parsing.
    """
    cleaned = text.strip()
    
    # Remove markdown code fences if present
    if cleaned.startswith("```"):
        # Find the end of the first line (which might be ```json)
        first_newline = cleaned.index("\n")
        # Find the closing fence
        last_fence = cleaned.rfind("```")
        cleaned = cleaned[first_newline + 1:last_fence].strip()
    
    return json.loads(cleaned)


def run_experiment(client, model, config, test_cases):
    """
    Run the full extraction pipeline on all test cases.
    
    Returns a list of result dicts, one per test case.
    """
    results = []
    
    for i, tc in enumerate(test_cases, 1):
        print(f"\nProcessing {i}/{len(test_cases)}: {tc['name']}...", end=" ")
        
        # Step 1: Build the prompt
        prompt = build_prompt(config, tc["abstract"])
        
        # Step 2: Call the API (and time it)
        start = time.time()
        raw_response = call_llm(client, model, prompt)
        elapsed = time.time() - start
        
        # Step 3: Parse the JSON response
        try:
            extraction = parse_json_response(raw_response)
            parse_error = None
        except (json.JSONDecodeError, ValueError) as e:
            extraction = {}
            parse_error = str(e)
            print(f"PARSE ERROR: {e}")
        
        # Step 4: Score against expected
        scores = score_extraction(tc["expected"], extraction)
        correct = sum(scores.values())
        total = len(scores)
        
        print(f"{correct}/{total} fields correct ({elapsed:.1f}s)")
        
        results.append({
            "name": tc["name"],
            "pmid": tc["pmid"],
            "extraction": extraction,
            "scores": scores,
            "elapsed": elapsed,
            "parse_error": parse_error
        })
    
    return results

In [8]:
# Run the experiment — this makes 4 API calls
print("Running extraction pipeline...")
results = run_experiment(client, MODEL, CONFIG, TEST_CASES)
print("\nDone!")

Running extraction pipeline...

Processing 1/4: MR CLEAN — thrombectomy RCT... 6/6 fields correct (1.9s)

Processing 2/4: CHS — aspirin and stroke in the elderly... 6/6 fields correct (1.4s)

Processing 3/4: STICA — cathodal tDCS pilot trial... 5/6 fields correct (1.3s)

Processing 4/4: ESUS — anticoagulation vs antiplatelets meta-analysis... 5/6 fields correct (2.1s)

Done!


## 8. Interpreting Results

Let's look at how the model performed — overall, per test case, and per field.

In [9]:
# Compute summary statistics
all_scores = [r["scores"] for r in results]
overall_accuracy, per_field_accuracy = summarize_scores(all_scores)

print(f"Overall accuracy: {overall_accuracy:.0%}")
print(f"\n{'='*60}")

# Per-case breakdown
print("\nPer-case accuracy:")
for r in results:
    correct = sum(r["scores"].values())
    total = len(r["scores"])
    print(f"  {r['name']}: {correct}/{total} ({correct/total:.0%})")

print(f"\n{'='*60}")

# Per-field breakdown
print("\nPer-field accuracy:")
for field_name, accuracy in per_field_accuracy.items():
    status = "ALL CORRECT" if accuracy == 1.0 else f"{accuracy:.0%}"
    print(f"  {field_name}: {status}")

Overall accuracy: 92%


Per-case accuracy:
  MR CLEAN — thrombectomy RCT: 6/6 (100%)
  CHS — aspirin and stroke in the elderly: 6/6 (100%)
  STICA — cathodal tDCS pilot trial: 5/6 (83%)
  ESUS — anticoagulation vs antiplatelets meta-analysis: 5/6 (83%)


Per-field accuracy:
  study_type: ALL CORRECT
  sample_size: ALL CORRECT
  intervention: ALL CORRECT
  primary_outcome_direction: ALL CORRECT
  population_age_group: 50%
  single_center: ALL CORRECT


In [10]:
# Detailed view: what did the model extract vs. what we expected?
print("Detailed comparison (expected -> extracted):")
print(f"{'='*60}\n")

for r in results:
    print(f"{r['name']} (PMID {r['pmid']})")
    print(f"-" * 40)
    
    tc = next(tc for tc in TEST_CASES if tc["pmid"] == r["pmid"])
    
    for field_name, correct in r["scores"].items():
        expected_val = tc["expected"][field_name]
        actual_val = r["extraction"].get(field_name, "MISSING")
        mark = "OK" if correct else "WRONG"
        
        if correct:
            print(f"  {mark:>5}  {field_name}: {actual_val}")
        else:
            print(f"  {mark:>5}  {field_name}: {actual_val} (expected: {expected_val})")
    
    print()

Detailed comparison (expected -> extracted):

MR CLEAN — thrombectomy RCT (PMID 25517348)
----------------------------------------
     OK  study_type: RCT
     OK  sample_size: 500
     OK  intervention: thrombectomy
     OK  primary_outcome_direction: positive
     OK  population_age_group: not_specified
     OK  single_center: False

CHS — aspirin and stroke in the elderly (PMID 9596230)
----------------------------------------
     OK  study_type: cohort
     OK  sample_size: 5011
     OK  intervention: aspirin
     OK  primary_outcome_direction: negative
     OK  population_age_group: elderly
     OK  single_center: False

STICA — cathodal tDCS pilot trial (PMID 33866820)
----------------------------------------
     OK  study_type: RCT
     OK  sample_size: 45
     OK  intervention: cathodal_tDCS
     OK  primary_outcome_direction: negative
  WRONG  population_age_group: not_specified (expected: adult)
     OK  single_center: True

ESUS — anticoagulation vs antiplatelets meta-ana

### Questions to consider

Look at the results above and think about:

1. **Which fields did the model get right consistently?** These are the ones where the information is usually stated explicitly — `study_type` and `sample_size` are often verbatim in the abstract.

2. **Which fields did it struggle with?** Look at the "WRONG" entries — can you see why the model made the mistake it did?

3. **Pay special attention to `population_age_group`.** For the STICA and ESUS abstracts, we labeled the age group as `"adult"` even though neither abstract states ages explicitly. Our reasoning: stroke patients receiving reperfusion therapy or enrolled in stroke prevention trials are adults — the clinical context makes this clear. But the model might answer `"not_specified"` because the age isn't literally stated. Both answers are defensible. This is exactly the kind of judgment call that makes extraction hard — and why you need domain experts reviewing the ground truth, not just the model's output.

4. **Would you change any ground truth labels?** Sometimes reviewing model outputs reveals that our "correct" answer was actually debatable. That's valuable information — it means the field definition needs tightening, not that the model is wrong.

5. **What would happen with more test cases?** Four abstracts give us a sense of the pattern, but you'd want 20–50 to have confidence in accuracy estimates for a real study.

## 9. What's Next

You've just built a working data extraction pipeline. The pattern you learned — **config defines what, code handles how, test cases verify correctness** — is the same one used in production research pipelines. Here's how it scales:

| This workbook | Production pipeline |
|--------------|--------------------|
| Config in a Python dict | Config in external YAML files |
| 4 hardcoded abstracts | Hundreds of documents fetched from PubMed API |
| 6 exact-match fields | Mix of exact-match, fuzzy, and human-reviewed fields |
| 1 model | Multiple models compared in a tournament |
| No cost tracking | Cost per extraction tracked and budgeted |
| Manual review | Automated quality checks with human review for edge cases |

### Suggested next steps

1. **Modify the config** — add a new field or change the valid values. Re-run the notebook and see how accuracy changes.
2. **Add more test cases** — find a PubMed abstract from your own research area and add it with your own ground truth.
3. **Try a different model** — change the `MODEL` variable in the Setup cell and compare results.
4. **Read about craft's LLM Extractor method** — this workbook teaches the pattern; craft provides the framework for using it in a reproducible study.